# 💳 Fraud Detection — SMOTE, Logistic Regression & Random Forest

**Oasis Infobyte Data Analytics — Level 2, Task 3**

This notebook follows a reader-friendly **Question → Code → Visual → Interpretation** structure. Each important graph is generated directly underneath its analysis code, followed immediately by a written interpretation so the reasoning is easy to follow from data exploration through model evaluation.

## 1. 📥 Load and Inspect the Dataset

We first load the credit-card transaction data and measure the class imbalance before making modelling decisions.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE

DATA_PATH='../data/creditcard.csv'
RESULTS_DIR='../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
df=pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('Class counts:')
print(df['Class'].value_counts())
print(f'Fraud percentage: {df["Class"].mean()*100:.4f}%')

### Interpretation

The original dataset contains **284,807 transactions**, of which only **492 are fraudulent (0.1727%)**. Fraud is therefore extremely rare, which means class imbalance must be addressed before evaluating model performance.

## 2. 📊 Class Distribution

How severe is the imbalance between legitimate and fraudulent transactions?

In [2]:
class_counts=df['Class'].value_counts().rename(index={0:'Legitimate',1:'Fraud'})
display(class_counts)
plt.figure(figsize=(7,4.5))
sns.barplot(x=class_counts.index,y=class_counts.values)
plt.title('Transaction Class Distribution')
plt.xlabel('Transaction Type'); plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'class_distribution.png'),dpi=160)
plt.show()

### 🔎 Interpretation

The graph shows an extreme imbalance: legitimate transactions vastly outnumber fraudulent ones. A classifier could achieve deceptively high accuracy by predicting almost every transaction as legitimate, while detecting very little fraud. This is why the project prioritises **Precision, Recall, F1-score and ROC-AUC** rather than accuracy alone.

## 3. 💳 Transaction Amount Analysis

Do fraudulent transactions have a different transaction-amount distribution from legitimate transactions? Because transaction amounts are highly skewed, a log transformation is used for the visual comparison.

In [3]:
amount_eda=df[['Amount','Class']].copy()
amount_eda['AmountLog']=np.log1p(amount_eda['Amount'])
plt.figure(figsize=(9,5))
sns.histplot(data=amount_eda,x='AmountLog',hue='Class',bins=50,element='step',stat='density',common_norm=False)
plt.title('Transaction Amount Distribution: Fraud vs Legitimate')
plt.xlabel('log(1 + Transaction Amount)'); plt.ylabel('Density')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'amount_distribution.png'),dpi=160)
plt.show()
print('Median legitimate amount:',df.loc[df['Class']==0,'Amount'].median())
print('Median fraudulent amount:',df.loc[df['Class']==1,'Amount'].median())

### 🔎 Interpretation

The distributions overlap, so transaction amount alone cannot reliably distinguish fraud from legitimate activity. The median legitimate amount is **22.00**, while the median fraudulent amount is **9.25**. This supports using the anonymised transaction features together with `Amount` rather than treating amount as a standalone fraud rule.

## 4. 🕐 Relative Time-of-Day Analysis

Does the fraud rate change across the dataset's 24-hour cycle? The `Time` column records elapsed seconds from the first transaction, so the resulting hour is **relative**, not a real-world clock time.

In [4]:
time_eda=df[['Time','Class']].copy()
time_eda['Hour']=(time_eda['Time']/3600)%24
time_eda['Hour']=time_eda['Hour'].astype(int)
hourly_counts=time_eda.groupby(['Hour','Class']).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
hourly_counts.columns=['Legitimate','Fraud']
hourly_counts['Total']=hourly_counts.sum(axis=1)
hourly_counts['FraudRatePct']=hourly_counts['Fraud']/hourly_counts['Total']*100
plt.figure(figsize=(10,5))
plt.plot(hourly_counts.index,hourly_counts['FraudRatePct'],marker='o')
plt.xticks(range(24))
plt.title('Fraud Rate by Relative Hour of Day')
plt.xlabel('Relative Hour'); plt.ylabel('Fraud Rate (%)')
plt.grid(alpha=.25)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fraud_rate_by_hour.png'),dpi=160)
plt.show()
display(hourly_counts[['Legitimate','Fraud','FraudRatePct']].round(4))

### 🔎 Interpretation

The fraud rate varies across the relative 24-hour cycle rather than remaining perfectly constant. These differences provide useful exploratory signal, but the `Time` feature does **not** represent a real-world timestamp, so the pattern should not be interpreted as a specific day/night or business-hour effect.

## 5. ⚖️ Stratified Train/Test Split and SMOTE

How do we handle the severe class imbalance without leaking information from the test set into training?

In [5]:
X=df.drop(columns='Class')
y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)
smote=SMOTE(random_state=42,sampling_strategy=.10)
X_smote,y_smote=smote.fit_resample(X_train_s,y_train)
print('Train shape:',X_train.shape)
print('Test shape:',X_test.shape)
print('Before SMOTE:',y_train.value_counts().to_dict())
print('After SMOTE:',y_smote.value_counts().to_dict())

### 🔎 Interpretation

The data is split using **stratification**, so fraud remains represented in both sets. `StandardScaler` is fitted only on the training data, and **SMOTE is applied only to the training set**. The test set remains untouched, giving us a fair evaluation on the original class distribution and preventing test-set leakage.

## 6. 🤖 Train and Compare the Models

We compare a linear baseline (Logistic Regression) with a nonlinear ensemble model (Random Forest). Both models are trained using the SMOTE-balanced training data.

In [6]:
models={
    'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),
    'Random Forest':RandomForestClassifier(n_estimators=20,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=42)
}
metrics=[]; predictions={}; probabilities={}
for name,model in models.items():
    model.fit(X_smote,y_smote)
    pred=model.predict(X_test_s)
    prob=model.predict_proba(X_test_s)[:,1]
    predictions[name]=pred
    probabilities[name]=prob
    metrics.append({'Model':name,'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob)})
metrics_df=pd.DataFrame(metrics)
display(metrics_df.style.format({c:'{:.4f}' for c in ['Precision','Recall','F1','ROC-AUC']}))
metrics_df.to_csv(os.path.join(RESULTS_DIR,'model_metrics.csv'),index=False)

### 🔎 Interpretation

**Random Forest is the strongest overall model in this run.** It achieves **80.00% Precision, 85.71% Recall, 82.76% F1 and 97.78% ROC-AUC**. Logistic Regression has slightly higher recall (**88.78%**) but much lower precision (**35.22%**), meaning it catches slightly more fraud at the cost of substantially more false-positive alerts.

## 7. 🧩 Confusion Matrices

How many legitimate and fraudulent transactions does each model classify correctly or incorrectly?

In [7]:
for name,pred in predictions.items():
    print(f'{name} confusion matrix:')
    print(confusion_matrix(y_test,pred))
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,(name,pred) in zip(axes,predictions.items()):
    sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cbar=False,ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'confusion_matrices.png'),dpi=160)
plt.show()

### 🔎 Interpretation

The Random Forest confusion matrix shows **84 true positives and 14 false negatives**, while Logistic Regression identifies **87 true positives and 11 false negatives**. This explains why Logistic Regression has the slightly higher recall. However, Random Forest produces only **21 false positives**, compared with **160** for Logistic Regression, which explains its much stronger precision and overall operational balance.

## 8. 📈 ROC Curve

How well can each model separate fraudulent from legitimate transactions across different classification thresholds?

In [8]:
plt.figure(figsize=(7,5))
for name,prob in probabilities.items():
    fpr,tpr,_=roc_curve(y_test,prob)
    auc=roc_auc_score(y_test,prob)
    print(f'ROC-AUC — {name}: {auc:.4f}')
    plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.4f})')
plt.plot([0,1],[0,1],'--',label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — SMOTE Models')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'roc_curve.png'),dpi=160)
plt.show()

### 🔎 Interpretation

Both models perform far better than random classification. Random Forest has the higher **ROC-AUC of 0.9778**, compared with **0.9674** for Logistic Regression, indicating stronger overall ranking performance across classification thresholds.

## 9. 🎯 Precision vs Recall Trade-off

Fraud detection is not simply about maximising one metric. Missing fraud and generating false alerts both have costs.

### Interpretation

Logistic Regression prioritises fraud capture in this experiment, with **88.78% recall**, but its **35.22% precision** means many alerts are false positives. Random Forest sacrifices a small amount of recall (**85.71%**) to achieve much stronger precision (**80.00%**) and F1 (**82.76%**).

For a production system, the classification threshold should be selected using the business cost of missed fraud versus the operational cost of investigating false alerts.

## 10. 🔎 Random Forest Feature Importance

Which anonymised features contribute most strongly to the Random Forest's fraud predictions?

In [9]:
rf=models['Random Forest']
importance=pd.DataFrame({'Feature':X.columns,'Importance':rf.feature_importances_}).sort_values('Importance',ascending=False)
importance.to_csv(os.path.join(RESULTS_DIR,'random_forest_feature_importance.csv'),index=False)
display(importance.head(10).round(6))
top=importance.head(10).sort_values('Importance')
plt.figure(figsize=(8,6))
plt.barh(top['Feature'],top['Importance'])
plt.xlabel('Importance')
plt.title('Top Fraud Features — Random Forest')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'feature_importance.png'),dpi=160)
plt.show()

### 🔎 Interpretation

The most influential features are **V14, V17, V12, V10 and V3**, followed by V16, V4, V9, V2 and V7. These `V` variables are anonymised PCA-derived features, so their importance represents predictive signal rather than directly interpretable business variables such as merchant category or customer demographics.

## 11. 🚀 Scalability — 1 Million Transactions per Hour

A production fraud system must be able to score transactions continuously and reliably at approximately **277.8 transactions per second**.

### Interpretation

At this scale, a production architecture should use streaming or micro-batched ingestion, efficient feature generation, parallel stateless scoring workers, load balancing, autoscaling, latency monitoring, data-quality checks, fraud-rate/model-drift monitoring, threshold management and periodic retraining. The exact infrastructure requirement should be established through production-like load testing rather than assumed from notebook execution time.

## ✅ Final Conclusion

The analysis demonstrates a complete fraud-detection workflow: severe class imbalance was quantified, fraud-focused EDA was performed, the data was split with stratification, scaling and SMOTE were applied without test leakage, and Logistic Regression was compared with Random Forest.

**Random Forest + SMOTE is the strongest overall model in this experiment**, balancing fraud detection with a substantially lower false-positive burden. Logistic Regression remains useful as a high-recall baseline.

The notebook is deliberately organised so that every major visual is followed immediately by its interpretation, making the analytical reasoning easier for an evaluator, recruiter or reader to understand.